<a href="https://colab.research.google.com/github/nhuvtq87/AAI2025/blob/2026fall/Agentic_AI_in_Customer_Service_and_Sales_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Customer Service and Sales Chatbot
**Multi-agent chatbot using LangGraph + Google Gemini API**

Based on starter code: `code_06_XX Multi-agent chatbots with routing`

**Agents:**
- **Product Agent** — laptop features & pricing
- **Orders Agent** — order status, quantity updates & refund policy
- **Small Talk Agent** — greetings and goodbyes
- **Router Agent** — classifies intent and routes to the right agent

**Memory:** LangGraph `MemorySaver` maintains conversation state across 3+ turns

## Setup API Key


In [4]:
from google.colab import userdata
import os
os.environ["Gemini_API_Key"] = userdata.get("Gemini_API_Key")
print("API key loaded.")

API key loaded.


## Install Dependencies

In [5]:
!pip install -q langchain-google-genai langgraph langchain langchain-community pandas
print("Done installing.")

Done installing.


## Cell 1 — Setup Google Gemini
Replaces `AzureChatOpenAI` from starter code

In [6]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

# Retrieve key and set the environment variable that LangChain expects
gemini_key = userdata.get("Gemini_API_Key")
os.environ["GOOGLE_API_KEY"] = gemini_key

model = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0,
    api_key=gemini_key
)
print("Google Gemini model ready.")

Google Gemini model ready.


## Tools
Product and order tools using CSV data

In [8]:
import pandas as pd
from langchain_core.tools import tool

# 1. Load data
PRICING_FILE_PATH = "/content/sample_data/Laptop pricing.csv"
ORDERS_FILE_PATH = "/content/sample_data/Laptop Orders.csv"

product_pricing_df = pd.read_csv(PRICING_FILE_PATH)
product_orders_df = pd.read_csv(ORDERS_FILE_PATH)

# Standardize Order ID column to string to avoid int/str lookup mismatches
if "Order ID" in product_orders_df.columns:
    product_orders_df["Order ID"] = product_orders_df["Order ID"].astype(str)


# 2. Product tools
@tool
def get_laptop_info(laptop_name: str) -> str:
    """Returns specifications, features, price, and shipping details for a laptop given its name."""
    match_df = product_pricing_df[
        product_pricing_df["Name"].str.contains(laptop_name.strip(), case=False, na=False, regex=False)
    ]
    if match_df.empty:
        return f"No laptop found matching '{laptop_name}'."
    return str(match_df.iloc[0].to_dict())


# 3. Order management tools
@tool
def get_order_details(order_id: str) -> str:
    """Returns details about a laptop order given its order ID."""
    clean_id = str(order_id).strip()
    match_df = product_orders_df[product_orders_df["Order ID"] == clean_id]
    if match_df.empty:
        return f"No order found with ID '{order_id}'."
    return str(match_df.iloc[0].to_dict())


@tool
def update_quantity(order_id: str, new_quantity: int) -> str:
    """Updates the quantity of laptops for a given order ID and saves changes to disk."""
    global product_orders_df
    clean_id = str(order_id).strip()
    match_idx = product_orders_df.index[product_orders_df["Order ID"] == clean_id]

    if match_idx.empty:
        return f"No order found with ID '{order_id}'."

    product_orders_df.loc[match_idx, "Quantity Ordered"] = new_quantity

    # Persist changes back to CSV
    product_orders_df.to_csv(ORDERS_FILE_PATH, index=False)
    return f"Order {order_id} quantity updated to {new_quantity}."


# 4. Policy tools
@tool
def get_refund_policy(topic: str) -> str:
    """Returns the refund/return policy. Topic: general, damaged, wrong_item, or timeline."""
    policies = {
        "damaged": "Report damaged items within 7 days with photos for a full refund or replacement.",
        "wrong_item": "Wrong items will be collected for free and replaced immediately.",
        "timeline": "Refunds are processed within 5-7 business days after item is received.",
        "general": "Laptops can be returned within 30 days of delivery in original condition."
    }
    topic_clean = topic.lower()
    for key in ["damaged", "wrong_item", "timeline"]:
        if key in topic_clean:
            return policies[key]
    return policies["general"]


print("All tools ready.")

All tools ready.


## Base Agent Class
Reusable agent class using `StateGraph` — same pattern as `code_04`

In [9]:
import operator
from typing import Annotated, TypedDict

from langchain_core.messages import AIMessage, BaseMessage, SystemMessage, ToolMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph


class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]


class CustomerServiceAgent:

    def __init__(self, model, tools, system_prompt: str = "", debug: bool = False):
        self.system_prompt = system_prompt
        self.debug = debug

        # Map tools and bind to model
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

        # Build graph
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_llm)
        graph.add_node("tools", self.call_tools)
        graph.set_entry_point("llm")

        graph.add_conditional_edges(
            "llm",
            self.is_tool_call,
            {True: "tools", False: END}
        )
        graph.add_edge("tools", "llm")

        self.memory = MemorySaver()
        self.graph = graph.compile(checkpointer=self.memory)

    def call_llm(self, state: AgentState):
        messages = state["messages"]
        if self.system_prompt:
            messages = [SystemMessage(content=self.system_prompt)] + messages

        result = self.model.invoke(messages)
        return {"messages": [result]}

    def is_tool_call(self, state: AgentState) -> bool:
        last_message = state["messages"][-1]
        tool_calls = getattr(last_message, "tool_calls", None)
        return bool(tool_calls)

    def call_tools(self, state: AgentState):
        last_message = state["messages"][-1]
        tool_calls = getattr(last_message, "tool_calls", [])
        results = []

        for tc in tool_calls:
            tool_name = tc.get("name")
            tool_args = tc.get("args", {})
            call_id = tc.get("id")

            if tool_name not in self.tools:
                content = f"Error: Tool '{tool_name}' not found."
            else:
                try:
                    content = str(self.tools[tool_name].invoke(tool_args))
                except Exception as e:
                    content = f"Error executing tool '{tool_name}': {str(e)}"

            results.append(
                ToolMessage(
                    tool_call_id=call_id,
                    name=tool_name,
                    content=content
                )
            )

        return {"messages": results}


print("Base agent class ready.")

Base agent class ready.


## Product & Orders Agents
Two specialist agents, each with their own system prompt

In [11]:
import inspect

# 1. Product Agent
product_agent = CustomerServiceAgent(
    model=model,
    tools=[get_laptop_info],
    system_prompt=inspect.cleandoc("""
        You are a professional assistant specialized in laptops sold by our company.
        Use the available tools to answer questions about laptop specifications, features, and pricing.
        Be helpful, concise, and professional.
    """)
)

# 2. Orders Agent
orders_agent = CustomerServiceAgent(
    model=model,
    tools=[get_order_details, update_quantity],
    system_prompt=inspect.cleandoc("""
        You are a professional assistant managing customer laptop orders.
        Use tools to look up order details and update quantities.
        Do NOT reveal information about any order ID other than the one explicitly requested.
        Direct refund policy questions to the refund department if you cannot resolve them.
    """)
)

# 3. Refund Agent
refund_agent = CustomerServiceAgent(
    model=model,
    tools=[get_refund_policy],
    system_prompt=inspect.cleandoc("""
        You are a customer service assistant specializing in returns and refunds.
        Use get_refund_policy to accurately explain return guidelines, timelines, and damage procedures.
        Do not invent policies or state that a refund has been processed.
    """)
)

print("Product Agent, Orders Agent, and Refund Agent updated and ready.")

Product Agent, Orders Agent, and Refund Agent updated and ready.


## Router Agent
From `code_06` — routes each message to the correct specialist agent

In [12]:
import functools
import operator
from typing import Annotated, Literal, TypedDict
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END


# 1. State definitions
class RouterState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    next_node: str


# 2. Reusable node wrapper to bridge sub-agent graphs
def agent_node(state: RouterState, agent, name: str):
    """Executes a specialist agent graph and appends its response."""
    result = agent.graph.invoke(
        {"messages": state["messages"]},
        config={"configurable": {"thread_id": "session"}}
    )
    last_message = result["messages"][-1]
    return {
        "messages": [AIMessage(content=last_message.content, name=name)]
    }


# 3. Router class
class RouterAgent:
    def __init__(self, model, system_prompt: str, smalltalk_prompt: str, debug: bool = False):
        self.model = model
        self.system_prompt = system_prompt.strip()
        self.smalltalk_prompt = smalltalk_prompt.strip()
        self.debug = debug

    def route(self, state: RouterState) -> dict:
        messages = [SystemMessage(content=self.system_prompt)] + state["messages"]
        response = self.model.invoke(messages)
        decision = response.content.strip().upper()

        valid_destinations = {"PRODUCT", "ORDER", "REFUND", "SMALLTALK"}
        next_step = decision if decision in valid_destinations else "SMALLTALK"

        if self.debug:
            print(f"[Router Decision]: {next_step}")

        return {"next_node": next_step}

    def handle_smalltalk(self, state: RouterState) -> dict:
        messages = [SystemMessage(content=self.smalltalk_prompt)] + state["messages"]
        response = self.model.invoke(messages)
        return {"messages": [AIMessage(content=response.content, name="Router_Smalltalk")]}


# 4. Bind specialist nodes
product_node = functools.partial(agent_node, agent=product_agent, name="Product_Agent")
orders_node  = functools.partial(agent_node, agent=orders_agent,  name="Orders_Agent")
refund_node  = functools.partial(agent_node, agent=refund_agent,  name="Refund_Agent")


# 5. Initialize Router
router_agent = RouterAgent(
    model=model,
    system_prompt="""
You are a Router. Analyze the user's message and classify it into exactly one category:
SMALLTALK  - greetings, goodbyes, general inquiries, or casual chat
PRODUCT    - questions about laptop features, hardware specs, stock, or pricing
ORDER      - questions about existing orders, delivery status, or order quantity updates
REFUND     - questions about return policies, damaged items, replacement rules, or refund timelines

Respond with ONLY that category name.
""",
    smalltalk_prompt="""
You are a helpful customer service coordinator.
Respond courteously to greetings, goodbyes, or general questions.
Inform the customer that you can help with laptop product inquiries, order tracking/updates, and returns/refund policies.
""",
    debug=False
)

print("Router Agent and specialist nodes ready.")

Router Agent and specialist nodes ready.


## Multi-Turn Conversation Demo
Shows all 3 agent types with MemorySaver keeping context

In [13]:
import inspect

# 1. Product Agent
product_agent = CustomerServiceAgent(
    model=model,
    tools=[get_laptop_info],  # Or [get_laptop_price, get_laptop_features] if kept separate
    system_prompt=inspect.cleandoc("""
        You are a professional assistant specialized in laptops sold by our company.
        Use the available tools to answer questions about laptop specifications, features, and pricing.
        Be helpful, concise, and professional.
    """)
)

# 2. Orders Agent
orders_agent = CustomerServiceAgent(
    model=model,
    tools=[get_order_details, update_quantity],
    system_prompt=inspect.cleandoc("""
        You are a professional assistant managing customer laptop orders.
        Use tools to look up order details and update quantities.
        Do NOT reveal information about any order ID other than the one explicitly requested.
        Direct refund policy questions to the refund department if you cannot resolve them.
    """)
)

# 3. Refund Agent
refund_agent = CustomerServiceAgent(
    model=model,
    tools=[get_refund_policy],
    system_prompt=inspect.cleandoc("""
        You are a customer service assistant specializing in returns and refunds.
        Use get_refund_policy to accurately explain return guidelines, timelines, and damage procedures.
        Do not invent policies or state that a refund has been processed.
    """)
)

print("Product Agent, Orders Agent, and Refund Agent initialized.")

Product Agent, Orders Agent, and Refund Agent initialized.


In [18]:
import functools
import operator
import uuid
from typing import Annotated, Literal, TypedDict
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph


# Helper to handle string or list format of Gemini message content
def extract_text(content) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        text_parts = [
            item.get("text", "") if isinstance(item, dict) else str(item)
            for item in content
        ]
        return "".join(text_parts)
    return str(content)


# 1. State Definition
class SupervisorState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    next_node: str


# 2. Node wrapper for sub-agent graphs
def call_specialist(state: SupervisorState, agent, name: str) -> dict:
    """Executes a specialized sub-agent and returns its final response."""
    result = agent.graph.invoke({"messages": state["messages"]})
    last_msg = result["messages"][-1]
    content_str = extract_text(last_msg.content)
    return {"messages": [AIMessage(content=content_str, name=name)]}


# 3. Router logic
ROUTER_SYSTEM_PROMPT = """You are a customer service supervisor.
Analyze the user's latest inquiry in the context of the ongoing conversation.
Classify the intent into exactly one of these categories:
- SMALLTALK : Greetings, goodbyes, compliments, or general non-product chat.
- PRODUCT   : Questions about laptop specifications, features, models, comparisons, or prices.
- ORDER     : Inquiries about existing order IDs, order details, delivery, or modifying item quantities.
- REFUND    : Questions regarding return guidelines, warranties, damaged goods, or refund timelines.

Respond with ONLY the single category word."""

def router_node(state: SupervisorState) -> dict:
    messages = [SystemMessage(content=ROUTER_SYSTEM_PROMPT)] + state["messages"]
    response = model.invoke(messages)
    raw_response = extract_text(response.content).strip().upper()

    valid_routes = {"PRODUCT", "ORDER", "REFUND", "SMALLTALK"}
    target = "SMALLTALK"
    for route in valid_routes:
        if route in raw_response:
            target = route
            break

    return {"next_node": target}

def smalltalk_node(state: SupervisorState) -> dict:
    system_prompt = (
        "You are a helpful customer service coordinator. "
        "Politely answer greetings or general queries, and guide the user on what you can assist with: "
        "laptop product info, order management, and refund policies."
    )
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = model.invoke(messages)
    content_str = extract_text(response.content)
    return {"messages": [AIMessage(content=content_str, name="Coordinator")]}


# 4. Build and compile supervisor graph
workflow = StateGraph(SupervisorState)

workflow.add_node("router", router_node)
workflow.add_node("smalltalk", smalltalk_node)
workflow.add_node("product", functools.partial(call_specialist, agent=product_agent, name="Product_Agent"))
workflow.add_node("order", functools.partial(call_specialist, agent=orders_agent, name="Orders_Agent"))
workflow.add_node("refund", functools.partial(call_specialist, agent=refund_agent, name="Refund_Agent"))

workflow.set_entry_point("router")

workflow.add_conditional_edges(
    "router",
    lambda state: state["next_node"].lower(),
    {
        "smalltalk": "smalltalk",
        "product": "product",
        "order": "order",
        "refund": "refund",
    }
)

workflow.add_edge("smalltalk", END)
workflow.add_edge("product", END)
workflow.add_edge("order", END)
workflow.add_edge("refund", END)

# Compile graph with persistence
supervisor_graph = workflow.compile(checkpointer=MemorySaver())


# 5. Execute Multi-Agent Test Runner
user_inputs = [
    "Hello, how are you?",
    "Tell me about the SpectraBook S",
    "How much does it cost?",
    "Show me the details of order ORD-7311",
    "Can you add one more laptop to that order?",
    "What is the general refund policy for laptops?",
    "Compare GammaAir X and AlphaBook Pro prices",
    "Goodbye!",
]

session_thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": session_thread_id}}

print("\n" + "=" * 60)
print("   CUSTOMER SERVICE CHATBOT — MULTI-AGENT DEMO")
print("=" * 60)

for user_input in user_inputs:
    print(f"\n{'─' * 55}")
    print(f"USER  : {user_input}")
    try:
        response = supervisor_graph.invoke(
            {"messages": [HumanMessage(content=user_input)]},
            config=config
        )
        print(f"AGENT : {response['messages'][-1].content}")
    except Exception as e:
        print(f"AGENT error executing model call: {e}")

print("\n" + "=" * 60)
print("DEMO COMPLETE")
print("=" * 60)


   CUSTOMER SERVICE CHATBOT — MULTI-AGENT DEMO

───────────────────────────────────────────────────────
USER  : Hello, how are you?
AGENT : Hello! I'm doing well, thank you for asking. 

I'm here to help you with any questions or support you might need. Specifically, I can assist you with:

* **Laptop Product Information:** Details on specifications, features, recommendations, and availability.
* **Order Management:** Tracking shipments, modifying existing orders, and checking order status.
* **Refund Policies:** Guidance on returns, warranty coverage, and processing refunds.

How can I assist you today?

───────────────────────────────────────────────────────
USER  : Tell me about the SpectraBook S
AGENT : Here are the details for the **SpectraBook S**:

* **Price:** $2,499
* **Estimated Shipping Time:** 7 business days

If you have questions about specific features, configurations, or need assistance placing an order, please let me know!

────────────────────────────────────────────

## Cell 7 — Memory Test: 3-Turn Conversation
Shows MemorySaver retaining context across 3 turns

In [19]:
import uuid
from langchain_core.messages import HumanMessage

# Ensure sub-agents use a thread scoped to the parent conversation
def call_specialist(state: SupervisorState, agent, name: str) -> dict:
    """Executes specialist agent with full message history."""
    result = agent.graph.invoke(
        {"messages": state["messages"]}
    )
    last_msg = result["messages"][-1]
    return {"messages": [AIMessage(content=last_msg.content, name=name)]}


print("=" * 55)
print("MEMORY TEST — 3-Turn Conversation")
print("=" * 55)

memory_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

turns = [
    "What are the details of order ID ORD-6948?",
    "Can you increase the quantity of it to 5?",
    "Can you show me the updated details of the order?",
]

for turn in turns:
    print(f"\n{'─' * 50}")
    print(f"USER  : {turn}")
    try:
        # UPDATE: Call supervisor_graph directly instead of router_agent.graph
        result = supervisor_graph.invoke(
            {"messages": [HumanMessage(content=turn)]},
            config=memory_config
        )
        print(f"AGENT : {result['messages'][-1].content}")
    except Exception as e:
        print(f"AGENT error executing model call: {e}")
        break

print("\nMemorySaver execution process completed.")

MEMORY TEST — 3-Turn Conversation

──────────────────────────────────────────────────
USER  : What are the details of order ID ORD-6948?
AGENT : Here are the details for order ID **ORD-6948**:

- **Order ID:** ORD-6948
- **Product Ordered:** OmegaPro G17
- **Quantity Ordered:** 3
- **Delivery Date:** October 25, 2024

──────────────────────────────────────────────────
USER  : Can you increase the quantity of it to 5?
AGENT : The quantity for order **ORD-6948** has been successfully updated to 5.

──────────────────────────────────────────────────
USER  : Can you show me the updated details of the order?
AGENT : Here are the updated details for order ID **ORD-6948**:

- **Order ID:** ORD-6948
- **Product Ordered:** OmegaPro G17
- **Quantity Ordered:** 5
- **Delivery Date:** October 25, 2024

MemorySaver execution process completed.
